In [38]:
from pathlib import Path
from remotezip import RemoteZip
from collections import Counter
import pandas as pd
from IPython.display import Audio, display
import librosa
import numpy as np
from urllib.request import urlretrieve
import zipfile
import json

In [3]:
URL = "https://zenodo.org/records/15588220/files/mammalps_v1.zip"

In [4]:
with RemoteZip(URL) as z:
    names = z.namelist()
    infos = z.infolist()

print("Total entries:", len(names))

Total entries: 28775


In [5]:
# Top-level structure actually present in the ZIP
top_level = Counter(
    name.split("/")[0]
    for name in names
    if name
)

print("\nTOP-LEVEL CONTENT:")
for name, count in top_level.items():
    print(f"{name:30} {count}")


TOP-LEVEL CONTENT:
benchmark_1                    24269
benchmark_2                    2000
raw_videos                     2472
reference_scenes               31
events2files.json              1
raw_videos_mammalps_v1.csv     1
file_list.txt                  1


In [6]:
metadata_extensions = {".csv", ".json", ".txt"}

metadata_files = []

for info in infos:
    path = Path(info.filename)

    if path.suffix.lower() in metadata_extensions:
        metadata_files.append(
            (info.filename, info.file_size)
        )

print(f"Metadata-like files found: {len(metadata_files)}\n")

for name, size in metadata_files:
    print(f"{size / 1024:8.1f} KB   {name}")

Metadata-like files found: 15

   130.8 KB   benchmark_1/metadata/test.csv
   444.1 KB   benchmark_1/metadata/train.csv
    71.5 KB   benchmark_1/metadata/val.csv
     5.1 KB   benchmark_2/metadata/dtci_r11/test.csv
    20.0 KB   benchmark_2/metadata/dtci_r11/train.csv
     5.1 KB   benchmark_2/metadata/dtci_r14/test.csv
    20.0 KB   benchmark_2/metadata/dtci_r14/train.csv
     5.7 KB   benchmark_2/metadata/dtci_source_r11/test.csv
    22.1 KB   benchmark_2/metadata/dtci_source_r11/train.csv
     5.7 KB   benchmark_2/metadata/dtci_source_r14/test.csv
    22.1 KB   benchmark_2/metadata/dtci_source_r14/train.csv
     0.2 KB   reference_scenes/annotations/class_ids.json
    58.0 KB   events2files.json
   265.1 KB   raw_videos_mammalps_v1.csv
  1503.8 KB   file_list.txt


In [7]:
targets = [
    "file_list.txt",
    "raw_videos_mammalps_v1.csv",
    "events2files.json",
]

for target in targets:
    matches = [
        name for name in names
        if name.endswith(target)
    ]

    print(f"\n{target}:")
    for match in matches:
        print("  ", match)


file_list.txt:
   file_list.txt

raw_videos_mammalps_v1.csv:
   raw_videos_mammalps_v1.csv

events2files.json:
   events2files.json


In [8]:
b1_metadata = [
    (info.filename, info.file_size)
    for info in infos
    if "benchmark_1" in info.filename.lower()
    and Path(info.filename).suffix.lower() in {".csv", ".json", ".txt"}
]

print(f"Benchmark I metadata files: {len(b1_metadata)}")

for name, size in b1_metadata:
    print(f"{size / 1024:8.1f} KB   {name}")

Benchmark I metadata files: 3
   130.8 KB   benchmark_1/metadata/test.csv
   444.1 KB   benchmark_1/metadata/train.csv
    71.5 KB   benchmark_1/metadata/val.csv


In [10]:
META_DIR = Path("metadata")
META_DIR.mkdir(parents=True, exist_ok=True)

metadata_paths = [
    "benchmark_1/metadata/train.csv",
    "benchmark_1/metadata/val.csv",
    "benchmark_1/metadata/test.csv",
]

with RemoteZip(URL) as z:
    for remote_path in metadata_paths:
        local_path = META_DIR / Path(remote_path).name

        local_path.write_bytes(z.read(remote_path))

        print(f"Saved: {local_path}")

Saved: metadata\train.csv
Saved: metadata\val.csv
Saved: metadata\test.csv


In [11]:
for path in sorted(META_DIR.glob("*.csv")):
    print(f"\n--- {path.name} ---")

    with open(path, "r", encoding="utf-8") as f:
        for _ in range(5):
            print(f.readline().rstrip())


--- test.csv ---
video_path,start_s,end_s,activity,actions,species
S1_C3_E154_V0066_ID1_T1/S1_C3_E154_V0066_ID1_T1_c0.mp4,0.0,2.6333333333333333,foraging,walking;none,red_deer
S1_C3_E190_V0079_ID1_T1/S1_C3_E190_V0079_ID1_T1_c0.mp4,0.0,2.3333333333333335,foraging,walking;none,red_deer
S1_C3_E191_V0080_ID1_T1/S1_C3_E191_V0080_ID1_T1_c0.mp4,0.0,0.26666666666666666,foraging,walking;none,red_deer
S1_C3_E191_V0080_ID1_T1/S1_C3_E191_V0080_ID1_T1_c1.mp4,0.0,2.4,courtship,standing_head_up;vocalizing,red_deer

--- train.csv ---
video_path,start_s,end_s,activity,actions,species
S1_C3_E144_V0060_ID1_T1/S1_C3_E144_V0060_ID1_T1_c0.mp4,0.0,4.966666666666667,camera_reaction,looking_at_camera;none,red_deer
S1_C3_E144_V0060_ID1_T1/S1_C3_E144_V0060_ID1_T1_c1.mp4,0.0,0.7666666666666667,camera_reaction,standing_head_up;none,red_deer
S1_C3_E144_V0060_ID1_T1/S1_C3_E144_V0060_ID1_T1_c2.mp4,0.0,4.0,camera_reaction,walking;none,red_deer
S1_C3_E145_V0061_ID1_T1/S1_C3_E145_V0061_ID1_T1_c0.mp4,0.0,3.5,foraging,wa

In [13]:
train_df = pd.read_csv(META_DIR / "train.csv")
val_df   = pd.read_csv(META_DIR / "val.csv")
test_df  = pd.read_csv(META_DIR / "test.csv")

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

Train: (4205, 6)
Val:   (686, 6)
Test:  (1244, 6)


In [14]:
print("Species:")
print(train_df["species"].value_counts())

print("\nActivities:")
print(train_df["activity"].value_counts())

print("\nActions:")
print(train_df["actions"].value_counts().head(20))

Species:
species
red_deer    4037
roe_deer     147
fox           10
hare           6
wolf           5
Name: count, dtype: int64

Activities:
activity
foraging           2302
vigilance           984
unknown             350
marking             154
grooming            148
courtship           129
resting              43
playing              32
escaping             29
camera_reaction      27
chasing               7
Name: count, dtype: int64

Actions:
actions
walking;none                   1013
standing_head_up;none           863
standing_head_down;none         757
grazing;none                    501
unknown;none                    209
running;none                    187
sniffing;none                   174
scratching_antlers;none          79
standing_head_up;vocalizing      77
scratching_body;none             58
scratching_hoof;none             51
laying;none                      42
drinking;none                    38
bathing;none                     36
shaking_fur;none                 21
lo

In [15]:
for name, df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    vocalizing = df["actions"].str.contains(
        "vocalizing",
        case=False,
        na=False
    )

    print(
        name,
        f"{vocalizing.sum()} / {len(df)}",
        f"({vocalizing.mean():.2%})"
    )

train 98 / 4205 (2.33%)
val 4 / 686 (0.58%)
test 37 / 1244 (2.97%)


In [16]:
all_metadata = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="val"),
        test_df.assign(split="test")
    ],
    ignore_index=True
)

all_metadata["vocalizing"] = (
    all_metadata["actions"]
    .str.contains("vocalizing", case=False, na=False)
)

vocal_samples = all_metadata[
    all_metadata["vocalizing"]
].sample(
    n=10,
    random_state=42
)

non_vocal_samples = all_metadata[
    ~all_metadata["vocalizing"]
].sample(
    n=10,
    random_state=42
)

inspection_samples = pd.concat(
    [vocal_samples, non_vocal_samples]
)

inspection_samples[
    ["video_path", "species", "activity", "actions"]
]

,video_path,species,activity,actions
5700,S3_C2_E709_V0241_ID1_T1/S3_C2_E709_V0241_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
2596,S3_C2_E799_V0535_ID1_T1/S3_C2_E799_V0535_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
2094,S3_C2_E730_V0322_ID1_T1/S3_C2_E730_V0322_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
5496,S3_C2_E670_V0158_ID1_T1/S3_C2_E670_V0158_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
2321,S3_C2_E765_V0426_ID1_T1/S3_C2_E765_V0426_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
835,S1_C2_E104_V0120_ID1_T1/S1_C2_E104_V0120_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
2436,S3_C2_E769_V0437_ID1_T1/S3_C2_E769_V0437_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
2612,S3_C2_E800_V0536_ID1_T1/S3_C2_E800_V0536_ID1_T...,red_deer,courtship,standing_head_up;vocalizing
5578,S3_C2_E706_V0197_ID1_T1/S3_C2_E706_V0197_ID1_T...,red_deer,courtship,laying;vocalizing
2525,S3_C2_E780_V0470_ID1_T1/S3_C2_E780_V0470_ID1_T...,red_deer,courtship,standing_head_up;vocalizing


In [17]:
AUDIO_DIR = Path("raw") / "inspection"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Index the actual Benchmark 1 WAV files in the archive
audio_index = {
    Path(name).stem: name
    for name in names
    if name.startswith("benchmark_1/audios/")
    and name.lower().endswith(".wav")
}

print("Audio files available:", len(audio_index))

Audio files available: 6135


In [18]:
inspection_samples = inspection_samples.copy()

inspection_samples["clip_id"] = (
    inspection_samples["video_path"]
    .apply(lambda x: Path(x).stem)
)

inspection_samples["audio_path"] = (
    inspection_samples["clip_id"]
    .map(audio_index)
)

print(
    "Matched:",
    inspection_samples["audio_path"].notna().sum(),
    "/",
    len(inspection_samples)
)

inspection_samples[
    ["clip_id", "species", "activity", "actions", "audio_path"]
]

Matched: 20 / 20


,clip_id,species,activity,actions,audio_path
5700,S3_C2_E709_V0241_ID1_T1_c1,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E709_V0241_ID1_T1/S3_...
2596,S3_C2_E799_V0535_ID1_T1_c3,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E799_V0535_ID1_T1/S3_...
2094,S3_C2_E730_V0322_ID1_T1_c0,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E730_V0322_ID1_T1/S3_...
5496,S3_C2_E670_V0158_ID1_T1_c6,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E670_V0158_ID1_T1/S3_...
2321,S3_C2_E765_V0426_ID1_T1_c0,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E765_V0426_ID1_T1/S3_...
835,S1_C2_E104_V0120_ID1_T1_c3,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S1_C2_E104_V0120_ID1_T1/S1_...
2436,S3_C2_E769_V0437_ID1_T1_c0,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E769_V0437_ID1_T1/S3_...
2612,S3_C2_E800_V0536_ID1_T1_c4,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E800_V0536_ID1_T1/S3_...
5578,S3_C2_E706_V0197_ID1_T1_c1,red_deer,courtship,laying;vocalizing,benchmark_1/audios/S3_C2_E706_V0197_ID1_T1/S3_...
2525,S3_C2_E780_V0470_ID1_T1_c4,red_deer,courtship,standing_head_up;vocalizing,benchmark_1/audios/S3_C2_E780_V0470_ID1_T1/S3_...


In [19]:
with RemoteZip(URL) as z:
    for _, row in inspection_samples.iterrows():
        remote_path = row["audio_path"]

        if pd.isna(remote_path):
            continue

        local_path = AUDIO_DIR / Path(remote_path).name

        local_path.write_bytes(z.read(remote_path))

        print("Saved:", local_path)

Saved: raw\inspection\S3_C2_E709_V0241_ID1_T1_c1.wav
Saved: raw\inspection\S3_C2_E799_V0535_ID1_T1_c3.wav
Saved: raw\inspection\S3_C2_E730_V0322_ID1_T1_c0.wav
Saved: raw\inspection\S3_C2_E670_V0158_ID1_T1_c6.wav
Saved: raw\inspection\S3_C2_E765_V0426_ID1_T1_c0.wav
Saved: raw\inspection\S1_C2_E104_V0120_ID1_T1_c3.wav
Saved: raw\inspection\S3_C2_E769_V0437_ID1_T1_c0.wav
Saved: raw\inspection\S3_C2_E800_V0536_ID1_T1_c4.wav
Saved: raw\inspection\S3_C2_E706_V0197_ID1_T1_c1.wav
Saved: raw\inspection\S3_C2_E780_V0470_ID1_T1_c4.wav
Saved: raw\inspection\S1_C1_E104_V0274_ID1_T1_c0.wav
Saved: raw\inspection\S2_C1_E323_V0073_ID1_T1_c2.wav
Saved: raw\inspection\S1_C1_E94_V0219_ID1_T1_c0.wav
Saved: raw\inspection\S3_C2_E407_V0022_ID1_T1_c0.wav
Saved: raw\inspection\S1_C1_E104_V0268_ID1_T1_c0.wav
Saved: raw\inspection\S3_C3_E693_V0706_ID3_T1_c4.wav
Saved: raw\inspection\S1_C1_E37_V0075_ID1_T1_c3.wav
Saved: raw\inspection\S3_C2_E732_V0341_ID1_T1_c0.wav
Saved: raw\inspection\S2_C3_E285_V0051_ID3_T1_c2

In [20]:
inspection_samples.to_csv(
    AUDIO_DIR / "inspection_metadata.csv",
    index=False
)

In [21]:
AUDIO_DIR = Path("raw") / "inspection"

inspection_df = pd.read_csv(
    AUDIO_DIR / "inspection_metadata.csv"
)

inspection_df["local_audio"] = inspection_df["clip_id"].apply(
    lambda x: AUDIO_DIR / f"{x}.wav"
)

inspection_df[
    ["clip_id", "species", "activity", "actions", "vocalizing"]
]

,clip_id,species,activity,actions,vocalizing
0,S3_C2_E709_V0241_ID1_T1_c1,red_deer,courtship,standing_head_up;vocalizing,True
1,S3_C2_E799_V0535_ID1_T1_c3,red_deer,courtship,standing_head_up;vocalizing,True
2,S3_C2_E730_V0322_ID1_T1_c0,red_deer,courtship,standing_head_up;vocalizing,True
3,S3_C2_E670_V0158_ID1_T1_c6,red_deer,courtship,standing_head_up;vocalizing,True
4,S3_C2_E765_V0426_ID1_T1_c0,red_deer,courtship,standing_head_up;vocalizing,True
5,S1_C2_E104_V0120_ID1_T1_c3,red_deer,courtship,standing_head_up;vocalizing,True
6,S3_C2_E769_V0437_ID1_T1_c0,red_deer,courtship,standing_head_up;vocalizing,True
7,S3_C2_E800_V0536_ID1_T1_c4,red_deer,courtship,standing_head_up;vocalizing,True
8,S3_C2_E706_V0197_ID1_T1_c1,red_deer,courtship,laying;vocalizing,True
9,S3_C2_E780_V0470_ID1_T1_c4,red_deer,courtship,standing_head_up;vocalizing,True


In [23]:
for vocalizing in [True, False]:

    print("\nVOCALIZING =", vocalizing)

    samples = inspection_df[
        inspection_df["vocalizing"] == vocalizing
    ].head(5)

    for _, row in samples.iterrows():
        print(
            row["clip_id"],
            "|",
            row["activity"],
            "|",
            row["actions"]
        )

        display(Audio(filename=str(row["local_audio"])))


VOCALIZING = True
S3_C2_E709_V0241_ID1_T1_c1 | courtship | standing_head_up;vocalizing


S3_C2_E799_V0535_ID1_T1_c3 | courtship | standing_head_up;vocalizing


S3_C2_E730_V0322_ID1_T1_c0 | courtship | standing_head_up;vocalizing


S3_C2_E670_V0158_ID1_T1_c6 | courtship | standing_head_up;vocalizing


S3_C2_E765_V0426_ID1_T1_c0 | courtship | standing_head_up;vocalizing



VOCALIZING = False
S1_C1_E104_V0274_ID1_T1_c0 | foraging | grazing;none


S2_C1_E323_V0073_ID1_T1_c2 | foraging | walking;none


S1_C1_E94_V0219_ID1_T1_c0 | foraging | grazing;none


S3_C2_E407_V0022_ID1_T1_c0 | camera_reaction | running;none


S1_C1_E104_V0268_ID1_T1_c0 | foraging | grazing;none


In [26]:
rows = []

for _, row in inspection_df.iterrows():

    y, sr = librosa.load(
        row["local_audio"],
        sr=None,
        mono=True
    )

    rows.append({
        "clip_id": row["clip_id"],
        "vocalizing": row["vocalizing"],
        "duration": len(y) / sr,
        "rms": np.sqrt(np.mean(y ** 2)),
        "zero_crossing_rate":
            librosa.feature.zero_crossing_rate(y).mean(),
        "spectral_centroid":
            librosa.feature.spectral_centroid(
                y=y,
                sr=sr
            ).mean()
    })

audio_stats = pd.DataFrame(rows)

audio_stats

,clip_id,vocalizing,duration,rms,zero_crossing_rate,spectral_centroid
0,S3_C2_E709_V0241_ID1_T1_c1,True,7.133333,0.038927,0.069543,1924.300598
1,S3_C2_E799_V0535_ID1_T1_c3,True,6.266667,0.011230,0.066953,1753.099060
2,S3_C2_E730_V0322_ID1_T1_c0,True,2.400000,0.014659,0.066892,1846.115445
3,S3_C2_E670_V0158_ID1_T1_c6,True,3.400000,0.020069,0.059659,1740.854266
4,S3_C2_E765_V0426_ID1_T1_c0,True,5.266667,0.067751,0.074027,2046.146112
5,S1_C2_E104_V0120_ID1_T1_c3,True,3.733333,0.000698,0.065859,2475.689182
6,S3_C2_E769_V0437_ID1_T1_c0,True,7.200000,0.017249,0.072708,2054.540813
7,S3_C2_E800_V0536_ID1_T1_c4,True,10.133333,0.019059,0.068488,1831.619856
8,S3_C2_E706_V0197_ID1_T1_c1,True,2.300000,0.011062,0.049392,1532.355450
9,S3_C2_E780_V0470_ID1_T1_c4,True,5.333333,0.016018,0.059411,1658.954970


In [27]:
audio_stats.groupby("vocalizing")[
    ["duration", "rms", "zero_crossing_rate", "spectral_centroid"]
].mean()

,duration,rms,zero_crossing_rate,spectral_centroid
vocalizing,,,,
False,5.150000,0.001482,0.072448,2450.512902
True,5.316667,0.021672,0.065293,1886.367575


In [28]:
B2_META_DIR = Path("metadata") / "benchmark_2"
DATASET_META_DIR = Path("metadata") / "dataset"

B2_META_DIR.mkdir(parents=True, exist_ok=True)
DATASET_META_DIR.mkdir(parents=True, exist_ok=True)

b2_metadata_paths = [
    name for name in names
    if name.startswith("benchmark_2/metadata/")
    and name.endswith(".csv")
]

root_metadata_paths = [
    "raw_videos_mammalps_v1.csv",
    "events2files.json",
]

print("Benchmark 2 metadata:")
for path in b2_metadata_paths:
    print(" ", path)

Benchmark 2 metadata:
  benchmark_2/metadata/dtci_r11/test.csv
  benchmark_2/metadata/dtci_r11/train.csv
  benchmark_2/metadata/dtci_r14/test.csv
  benchmark_2/metadata/dtci_r14/train.csv
  benchmark_2/metadata/dtci_source_r11/test.csv
  benchmark_2/metadata/dtci_source_r11/train.csv
  benchmark_2/metadata/dtci_source_r14/test.csv
  benchmark_2/metadata/dtci_source_r14/train.csv


In [29]:
with RemoteZip(URL) as z:

    for remote_path in b2_metadata_paths:
        parts = Path(remote_path).parts

        local_name = "_".join(parts[-3:])

        local_path = B2_META_DIR / local_name
        local_path.write_bytes(z.read(remote_path))

        print("Saved:", local_path)

    for remote_path in root_metadata_paths:
        local_path = DATASET_META_DIR / Path(remote_path).name
        local_path.write_bytes(z.read(remote_path))

        print("Saved:", local_path)

Saved: metadata\benchmark_2\metadata_dtci_r11_test.csv
Saved: metadata\benchmark_2\metadata_dtci_r11_train.csv
Saved: metadata\benchmark_2\metadata_dtci_r14_test.csv
Saved: metadata\benchmark_2\metadata_dtci_r14_train.csv
Saved: metadata\benchmark_2\metadata_dtci_source_r11_test.csv
Saved: metadata\benchmark_2\metadata_dtci_source_r11_train.csv
Saved: metadata\benchmark_2\metadata_dtci_source_r14_test.csv
Saved: metadata\benchmark_2\metadata_dtci_source_r14_train.csv
Saved: metadata\dataset\raw_videos_mammalps_v1.csv
Saved: metadata\dataset\events2files.json


In [30]:
for path in sorted(B2_META_DIR.glob("*.csv")):

    print(f"\n--- {path.name} ---")

    with open(path, "r", encoding="utf-8") as f:
        for _ in range(5):
            print(f.readline().rstrip())


--- metadata_dtci_r11_test.csv ---
E130 E130/E130_dtci_r11.npy courtship,foraging,unknown,vigilance red_deer overcast 1
E131 E131/E131_dtci_r11.npy foraging,unknown red_deer overcast 1
E132 E132/E132_dtci_r11.npy foraging red_deer rainy 1
E133 E133/E133_dtci_r11.npy foraging red_deer rainy 1
E134 E134/E134_dtci_r11.npy foraging,unknown red_deer overcast 1

--- metadata_dtci_r11_train.csv ---
E10 E10/E10_dtci_r11.npy foraging,unknown,vigilance roe_deer overcast 3+
E103 E103/E103_dtci_r11.npy foraging,unknown,vigilance red_deer clear 1
E104 E104/E104_dtci_r11.npy courtship,foraging,grooming,vigilance red_deer clear 1
E105 E105/E105_dtci_r11.npy courtship,foraging,marking,unknown,vigilance red_deer clear 1
E106 E106/E106_dtci_r11.npy foraging,unknown,vigilance red_deer clear 1

--- metadata_dtci_r14_test.csv ---
E130 E130/E130_dtci_r14.npy courtship,foraging,unknown,vigilance red_deer overcast 1
E131 E131/E131_dtci_r14.npy foraging,unknown red_deer overcast 1
E132 E132/E132_dtci_r14.npy 

In [32]:
DENSE_URL = (
    "https://zenodo.org/records/15588220/"
    "files/dense_annotations_v1.zip"
)

DENSE_ZIP = Path("metadata") / "dense_annotations_v1.zip"

if not DENSE_ZIP.exists():
    urlretrieve(DENSE_URL, DENSE_ZIP)

print(
    DENSE_ZIP,
    f"{DENSE_ZIP.stat().st_size / 1024 / 1024:.2f} MB"
)

metadata\dense_annotations_v1.zip 12.90 MB


In [35]:
with zipfile.ZipFile(DENSE_ZIP) as z:
    dense_names = z.namelist()

print("Entries:", len(dense_names))

print("\nFirst 20:")
for name in dense_names[:20]:
    print(name)

Entries: 2472

First 20:
dense_annotations/
dense_annotations/S1/
dense_annotations/S1/C1/
dense_annotations/S1/C1/S1_C1_E103_V0253.json
dense_annotations/S1/C1/S1_C1_E103_V0254.json
dense_annotations/S1/C1/S1_C1_E103_V0255.json
dense_annotations/S1/C1/S1_C1_E103_V0256.json
dense_annotations/S1/C1/S1_C1_E103_V0257.json
dense_annotations/S1/C1/S1_C1_E103_V0258.json
dense_annotations/S1/C1/S1_C1_E103_V0259.json
dense_annotations/S1/C1/S1_C1_E103_V0260.json
dense_annotations/S1/C1/S1_C1_E103_V0261.json
dense_annotations/S1/C1/S1_C1_E104_V0262.json
dense_annotations/S1/C1/S1_C1_E104_V0263.json
dense_annotations/S1/C1/S1_C1_E104_V0264.json
dense_annotations/S1/C1/S1_C1_E104_V0265.json
dense_annotations/S1/C1/S1_C1_E104_V0266.json
dense_annotations/S1/C1/S1_C1_E104_V0267.json
dense_annotations/S1/C1/S1_C1_E104_V0268.json
dense_annotations/S1/C1/S1_C1_E104_V0269.json


In [36]:
dense_json_files = [
    name for name in dense_names
    if name.lower().endswith(".json")
]

print("JSON annotations:", len(dense_json_files))

JSON annotations: 2459


In [39]:
with zipfile.ZipFile(DENSE_ZIP) as z:

    with z.open(dense_json_files[0]) as f:
        example = json.load(f)

print(example.keys())
print()
print(example["info"])
print()
print(example["frames"][0])

dict_keys(['info', 'frames'])

{'site_id': 'S1', 'cam_id': 'C1', 'event_id': 'E103', 'file_id': 'S1_C1_E103_V0253', 'file_path': 'S1/C1/S1_C1_E103_V0253.mp4', 'num_frames': 615, 'duration_s': 20.5, 'fps': 30.0, 'resolution': '1920x1080', 'automatically_discarded': False, 'attributes': {'meteo_conditions': 'none'}}

{'frame_id': 0, 'detections': []}


In [40]:
rows = []

with zipfile.ZipFile(DENSE_ZIP) as z:

    for json_path in dense_json_files:

        with z.open(json_path) as f:
            annotation = json.load(f)

        info = annotation["info"]
        frames = annotation["frames"]

        frames_with_detection = sum(
            1
            for frame in frames
            if len(frame["detections"]) > 0
        )

        total_detections = sum(
            len(frame["detections"])
            for frame in frames
        )

        rows.append({
            "file_id": info["file_id"],
            "event_id": info["event_id"],
            "site_id": info["site_id"],
            "cam_id": info["cam_id"],
            "duration_s": info["duration_s"],
            "fps": info["fps"],
            "automatically_discarded":
                info["automatically_discarded"],
            "num_frames": len(frames),
            "frames_with_detection":
                frames_with_detection,
            "total_detections":
                total_detections,
            "has_detection":
                frames_with_detection > 0,
        })

dense_summary = pd.DataFrame(rows)

dense_summary.head()

,file_id,event_id,site_id,cam_id,duration_s,fps,automatically_discarded,num_frames,frames_with_detection,total_detections,has_detection
0,S1_C1_E103_V0253,E103,S1,C1,20.5,30.0,False,615,614,614,True
1,S1_C1_E103_V0254,E103,S1,C1,20.5,30.0,False,615,615,615,True
2,S1_C1_E103_V0255,E103,S1,C1,11.5,30.0,False,345,335,335,True
3,S1_C1_E103_V0256,E103,S1,C1,4.0,30.0,False,120,119,119,True
4,S1_C1_E103_V0257,E103,S1,C1,8.0,30.0,False,240,228,228,True


In [41]:
pd.crosstab(
    dense_summary["automatically_discarded"],
    dense_summary["has_detection"],
    margins=True
)

has_detection,False,True,All
automatically_discarded,,,
False,0,1711,1711
True,748,0,748
All,748,1711,2459


In [42]:
dense_summary[
    [
        "duration_s",
        "frames_with_detection",
        "total_detections"
    ]
].describe()

,duration_s,frames_with_detection,total_detections
count,2459.000000,2459.000000,2459.000000
mean,22.434906,344.666531,387.741358
std,31.947447,571.985143,716.078156
min,0.466667,0.000000,0.000000
25%,1.500000,0.000000,0.000000
50%,13.500000,89.000000,90.000000
75%,20.500000,615.000000,615.000000
max,120.500000,3615.000000,8738.000000


In [43]:
dense_summary.to_csv(
    Path("metadata") / "dense_summary.csv",
    index=False
)

In [44]:
RAW_META = Path("metadata") / "dataset" / "raw_videos_mammalps_v1.csv"

raw_meta = pd.read_csv(RAW_META)

print(raw_meta.shape)
print(raw_meta.columns.tolist())

raw_meta.head()

(2459, 12)
['file_id', 'site_id', 'cam_id', 'video_id', 'event_id', 'duration', 'fps', 'resolution', 'date_time', 'file_path', 'in_B1', 'in_B2']


,file_id,site_id,cam_id,video_id,event_id,duration,fps,resolution,date_time,file_path,in_B1,in_B2
0,S1_C1_E103_V0253,S1,C1,V0253,E103,20.5,30.0,1920x1080,2023-09-09 23:22:52,S1/C1/S1_C1_E103_V0253.mp4,True,True
1,S1_C1_E103_V0254,S1,C1,V0254,E103,20.5,30.0,1920x1080,2023-09-09 23:23:16,S1/C1/S1_C1_E103_V0254.mp4,True,True
2,S1_C1_E103_V0255,S1,C1,V0255,E103,11.5,30.0,1920x1080,2023-09-09 23:23:32,S1/C1/S1_C1_E103_V0255.mp4,True,True
3,S1_C1_E103_V0256,S1,C1,V0256,E103,4.0,30.0,1920x1080,2023-09-09 23:23:38,S1/C1/S1_C1_E103_V0256.mp4,True,True
4,S1_C1_E103_V0257,S1,C1,V0257,E103,8.0,30.0,1920x1080,2023-09-09 23:23:54,S1/C1/S1_C1_E103_V0257.mp4,True,True


In [45]:
video_summary = raw_meta.merge(
    dense_summary,
    on="file_id",
    how="left",
    suffixes=("_raw", "_dense")
)

print(video_summary.shape)

video_summary.head()

(2459, 22)


,file_id,site_id_raw,cam_id_raw,video_id,event_id_raw,duration,fps_raw,resolution,date_time,file_path,...,event_id_dense,site_id_dense,cam_id_dense,duration_s,fps_dense,automatically_discarded,num_frames,frames_with_detection,total_detections,has_detection
0,S1_C1_E103_V0253,S1,C1,V0253,E103,20.5,30.0,1920x1080,2023-09-09 23:22:52,S1/C1/S1_C1_E103_V0253.mp4,...,E103,S1,C1,20.5,30.0,False,615,614,614,True
1,S1_C1_E103_V0254,S1,C1,V0254,E103,20.5,30.0,1920x1080,2023-09-09 23:23:16,S1/C1/S1_C1_E103_V0254.mp4,...,E103,S1,C1,20.5,30.0,False,615,615,615,True
2,S1_C1_E103_V0255,S1,C1,V0255,E103,11.5,30.0,1920x1080,2023-09-09 23:23:32,S1/C1/S1_C1_E103_V0255.mp4,...,E103,S1,C1,11.5,30.0,False,345,335,335,True
3,S1_C1_E103_V0256,S1,C1,V0256,E103,4.0,30.0,1920x1080,2023-09-09 23:23:38,S1/C1/S1_C1_E103_V0256.mp4,...,E103,S1,C1,4.0,30.0,False,120,119,119,True
4,S1_C1_E103_V0257,S1,C1,V0257,E103,8.0,30.0,1920x1080,2023-09-09 23:23:54,S1/C1/S1_C1_E103_V0257.mp4,...,E103,S1,C1,8.0,30.0,False,240,228,228,True


In [46]:
discarded = video_summary[
    video_summary["automatically_discarded"] == True
].copy()

print("Discarded videos:", len(discarded))

discarded[
    [
        "file_id",
        "event_id_raw",
        "site_id_raw",
        "cam_id_raw",
        "duration",
        "in_B1",
        "in_B2",
        "frames_with_detection"
    ]
].head(20)

Discarded videos: 748


,file_id,event_id_raw,site_id_raw,cam_id_raw,duration,in_B1,in_B2,frames_with_detection
19,S1_C1_E104_V0272,E104,S1,C1,1.5,True,True,0
22,S1_C1_E104_V0275,E104,S1,C1,1.5,True,True,0
39,S1_C1_E105_V0292,E105,S1,C1,1.5,True,True,0
57,S1_C1_E105_V0310,E105,S1,C1,4.0,True,True,0
59,S1_C1_E106_V0312,E106,S1,C1,1.5,True,True,0
60,S1_C1_E106_V0313,E106,S1,C1,1.5,True,True,0
61,S1_C1_E106_V0314,E106,S1,C1,1.5,True,True,0
64,S1_C1_E106_V0317,E106,S1,C1,1.0,True,True,0
65,S1_C1_E106_V0318,E106,S1,C1,1.0,True,True,0
66,S1_C1_E106_V0319,E106,S1,C1,1.0,True,True,0


In [47]:
print("\nBy site:")
print(discarded["site_id_raw"].value_counts())

print("\nBy camera:")
print(
    discarded.groupby(
        ["site_id_raw", "cam_id_raw"]
    ).size()
)

print("\nUsed in Benchmark 2:")
print(discarded["in_B2"].value_counts())

print("\nDuration:")
print(discarded["duration"].describe())


By site:
site_id_raw
S1    487
S3    242
S2     19
Name: count, dtype: int64

By camera:
site_id_raw  cam_id_raw
S1           C1            334
             C2            149
             C3              4
S2           C1             10
             C2              4
             C3              5
S3           C1             11
             C2             81
             C3            150
dtype: int64

Used in Benchmark 2:
in_B2
True     745
False      3
Name: count, dtype: int64

Duration:
count    748.000000
mean      19.652986
std       33.875862
min        0.466667
25%        1.000000
50%        1.000000
75%       20.500000
max      120.500000
Name: duration, dtype: float64


In [49]:
print(video_summary.columns.tolist())

['file_id', 'site_id_raw', 'cam_id_raw', 'video_id', 'event_id_raw', 'duration', 'fps_raw', 'resolution', 'date_time', 'file_path', 'in_B1', 'in_B2', 'event_id_dense', 'site_id_dense', 'cam_id_dense', 'duration_s', 'fps_dense', 'automatically_discarded', 'num_frames', 'frames_with_detection', 'total_detections', 'has_detection']


In [51]:
sample_negatives = pd.concat(
    [
        group.sample(
            n=min(3, len(group)),
            random_state=42
        )
        for _, group in discarded.groupby(
            ["site_id_raw", "cam_id_raw"]
        )
    ],
    ignore_index=True
)

print(sample_negatives.columns.tolist())

['file_id', 'site_id_raw', 'cam_id_raw', 'video_id', 'event_id_raw', 'duration', 'fps_raw', 'resolution', 'date_time', 'file_path', 'in_B1', 'in_B2', 'event_id_dense', 'site_id_dense', 'cam_id_dense', 'duration_s', 'fps_dense', 'automatically_discarded', 'num_frames', 'frames_with_detection', 'total_detections', 'has_detection']


In [52]:
sample_negatives[
    [
        "file_id",
        "event_id_raw",
        "site_id_raw",
        "cam_id_raw",
        "duration",
        "file_path",
        "in_B1",
        "in_B2",
        "automatically_discarded",
        "frames_with_detection"
    ]
]

,file_id,event_id_raw,site_id_raw,cam_id_raw,duration,file_path,in_B1,in_B2,automatically_discarded,frames_with_detection
0,S1_C1_E118_V0392,E118,S1,C1,1.0,S1/C1/S1_C1_E118_V0392.mp4,True,True,True,0
1,S1_C1_E210_V0851,E210,S1,C1,120.5,S1/C1/S1_C1_E210_V0851.mp4,False,True,True,0
2,S1_C1_E123_V0507,E123,S1,C1,1.0,S1/C1/S1_C1_E123_V0507.mp4,True,True,True,0
3,S1_C2_E198_V0484,E198,S1,C2,1.5,S1/C2/S1_C2_E198_V0484.mp4,True,True,True,0
4,S1_C2_E122_V0258,E122,S1,C2,1.0,S1/C2/S1_C2_E122_V0258.mp4,True,True,True,0
5,S1_C2_E243_V0541,E243,S1,C2,2.5,S1/C2/S1_C2_E243_V0541.mp4,True,True,True,0
6,S1_C3_E140_V0058,E140,S1,C3,120.5,S1/C3/S1_C3_E140_V0058.mp4,False,True,True,0
7,S1_C3_E67_V0033,E67,S1,C3,20.5,S1/C3/S1_C3_E67_V0033.mp4,True,True,True,0
8,S1_C3_E140_V0057,E140,S1,C3,120.5,S1/C3/S1_C3_E140_V0057.mp4,False,True,True,0
9,S2_C1_E322_V0070,E322,S2,C1,120.5,S2/C1/S2_C1_E322_V0070.mp4,False,True,True,0


In [53]:
print("Candidate negatives:", len(discarded))

print("\nBy site/camera:")
print(
    discarded.groupby(
        ["site_id_raw", "cam_id_raw"]
    ).size()
)

print("\nIncluded in Benchmark 2:")
print(discarded["in_B2"].value_counts(dropna=False))

print("\nIncluded in Benchmark 1:")
print(discarded["in_B1"].value_counts(dropna=False))

print("\nDuration:")
print(discarded["duration"].describe())

Candidate negatives: 748

By site/camera:
site_id_raw  cam_id_raw
S1           C1            334
             C2            149
             C3              4
S2           C1             10
             C2              4
             C3              5
S3           C1             11
             C2             81
             C3            150
dtype: int64

Included in Benchmark 2:
in_B2
True     745
False      3
Name: count, dtype: int64

Included in Benchmark 1:
in_B1
True     578
False    170
Name: count, dtype: int64

Duration:
count    748.000000
mean      19.652986
std       33.875862
min        0.466667
25%        1.000000
50%        1.000000
75%       20.500000
max      120.500000
Name: duration, dtype: float64


In [54]:
# Keep only one candidate video per event within each site/camera
diverse_discarded = (
    discarded
    .sort_values("duration", ascending=False)
    .drop_duplicates(
        subset=["site_id_raw", "cam_id_raw", "event_id_raw"]
    )
)

sample_negatives = pd.concat(
    [
        group.sample(
            n=min(3, len(group)),
            random_state=42
        )
        for _, group in diverse_discarded.groupby(
            ["site_id_raw", "cam_id_raw"]
        )
    ],
    ignore_index=True
)

sample_negatives[
    [
        "file_id",
        "event_id_raw",
        "site_id_raw",
        "cam_id_raw",
        "duration",
        "file_path",
        "in_B1",
        "in_B2"
    ]
]

,file_id,event_id_raw,site_id_raw,cam_id_raw,duration,file_path,in_B1,in_B2
0,S1_C1_E131_V0629,E131,S1,C1,1.0,S1/C1/S1_C1_E131_V0629.mp4,True,True
1,S1_C1_E154_V0778,E154,S1,C1,1.0,S1/C1/S1_C1_E154_V0778.mp4,True,True
2,S1_C1_E24_V0057,E24,S1,C1,120.5,S1/C1/S1_C1_E24_V0057.mp4,False,True
3,S1_C2_E238_V0537,E238,S1,C2,1.0,S1/C2/S1_C2_E238_V0537.mp4,True,True
4,S1_C2_E197_V0460,E197,S1,C2,1.5,S1/C2/S1_C2_E197_V0460.mp4,True,True
5,S1_C2_E251_V0570,E251,S1,C2,1.5,S1/C2/S1_C2_E251_V0570.mp4,True,True
6,S1_C3_E140_V0057,E140,S1,C3,120.5,S1/C3/S1_C3_E140_V0057.mp4,False,True
7,S1_C3_E228_V0090,E228,S1,C3,120.5,S1/C3/S1_C3_E228_V0090.mp4,False,True
8,S1_C3_E67_V0033,E67,S1,C3,20.5,S1/C3/S1_C3_E67_V0033.mp4,True,True
9,S2_C1_E322_V0068,E322,S2,C1,120.5,S2/C1/S2_C1_E322_V0068.mp4,False,True


In [55]:
# Build an index from actual MP4 entries in the archive
raw_video_index = {
    Path(name).stem: name
    for name in names
    if name.lower().endswith(".mp4")
}

sample_negatives["remote_path"] = (
    sample_negatives["file_id"]
    .map(raw_video_index)
)

print(
    "Matched:",
    sample_negatives["remote_path"].notna().sum(),
    "/",
    len(sample_negatives)
)

sample_negatives[
    [
        "file_id",
        "site_id_raw",
        "cam_id_raw",
        "event_id_raw",
        "duration",
        "remote_path"
    ]
]

Matched: 26 / 26


,file_id,site_id_raw,cam_id_raw,event_id_raw,duration,remote_path
0,S1_C1_E131_V0629,S1,C1,E131,1.0,raw_videos/S1/C1/S1_C1_E131_V0629.mp4
1,S1_C1_E154_V0778,S1,C1,E154,1.0,raw_videos/S1/C1/S1_C1_E154_V0778.mp4
2,S1_C1_E24_V0057,S1,C1,E24,120.5,raw_videos/S1/C1/S1_C1_E24_V0057.mp4
3,S1_C2_E238_V0537,S1,C2,E238,1.0,raw_videos/S1/C2/S1_C2_E238_V0537.mp4
4,S1_C2_E197_V0460,S1,C2,E197,1.5,raw_videos/S1/C2/S1_C2_E197_V0460.mp4
5,S1_C2_E251_V0570,S1,C2,E251,1.5,raw_videos/S1/C2/S1_C2_E251_V0570.mp4
6,S1_C3_E140_V0057,S1,C3,E140,120.5,raw_videos/S1/C3/S1_C3_E140_V0057.mp4
7,S1_C3_E228_V0090,S1,C3,E228,120.5,raw_videos/S1/C3/S1_C3_E228_V0090.mp4
8,S1_C3_E67_V0033,S1,C3,E67,20.5,raw_videos/S1/C3/S1_C3_E67_V0033.mp4
9,S2_C1_E322_V0068,S2,C1,E322,120.5,raw_videos/S2/C1/S2_C1_E322_V0068.mp4


In [56]:
NEGATIVE_INSPECTION_DIR = Path("raw") / "negative_inspection"
NEGATIVE_INSPECTION_DIR.mkdir(parents=True, exist_ok=True)

with RemoteZip(URL) as z:
    for _, row in sample_negatives.iterrows():
        remote_path = row["remote_path"]

        if pd.isna(remote_path):
            continue

        local_path = (
            NEGATIVE_INSPECTION_DIR /
            Path(remote_path).name
        )

        if not local_path.exists():
            local_path.write_bytes(
                z.read(remote_path)
            )

        print("Saved:", local_path)

Saved: raw\negative_inspection\S1_C1_E131_V0629.mp4
Saved: raw\negative_inspection\S1_C1_E154_V0778.mp4
Saved: raw\negative_inspection\S1_C1_E24_V0057.mp4
Saved: raw\negative_inspection\S1_C2_E238_V0537.mp4
Saved: raw\negative_inspection\S1_C2_E197_V0460.mp4
Saved: raw\negative_inspection\S1_C2_E251_V0570.mp4
Saved: raw\negative_inspection\S1_C3_E140_V0057.mp4
Saved: raw\negative_inspection\S1_C3_E228_V0090.mp4
Saved: raw\negative_inspection\S1_C3_E67_V0033.mp4
Saved: raw\negative_inspection\S2_C1_E322_V0068.mp4
Saved: raw\negative_inspection\S2_C1_E312_V0062.mp4
Saved: raw\negative_inspection\S2_C1_E292_V0045.mp4
Saved: raw\negative_inspection\S2_C2_E330_V0077.mp4
Saved: raw\negative_inspection\S2_C2_E375_V0118.mp4
Saved: raw\negative_inspection\S2_C2_E297_V0048.mp4
Saved: raw\negative_inspection\S2_C3_E265_V0022.mp4
Saved: raw\negative_inspection\S2_C3_E292_V0060.mp4
Saved: raw\negative_inspection\S2_C3_E270_V0032.mp4
Saved: raw\negative_inspection\S3_C1_E407_V0159.mp4
Saved: raw\neg

In [57]:
sample_negatives.to_csv(
    NEGATIVE_INSPECTION_DIR / "negative_inspection_metadata.csv",
    index=False
)

In [58]:
B2_META_DIR = Path("metadata") / "benchmark_2"

columns = [
    "event_id",
    "feature_path",
    "activities",
    "species",
    "weather",
    "num_individuals",
]

train_path = B2_META_DIR / "metadata_dtci_r11_train.csv"
test_path  = B2_META_DIR / "metadata_dtci_r11_test.csv"

b2_train = pd.read_csv(
    train_path,
    sep=r"\s+",
    header=None,
    names=columns,
    dtype=str
)

b2_test = pd.read_csv(
    test_path,
    sep=r"\s+",
    header=None,
    names=columns,
    dtype=str
)

b2 = pd.concat(
    [
        b2_train.assign(split="train"),
        b2_test.assign(split="test")
    ],
    ignore_index=True
)

print("Rows:", len(b2))

Rows: 397


In [59]:
print(b2.shape)
print()
print(b2.isna().sum())

(397, 7)

event_id           0
feature_path       0
activities         0
species            0
weather            0
num_individuals    0
split              0
dtype: int64


In [60]:
print("SPECIES")
print(b2["species"].value_counts(dropna=False))

SPECIES
species
red_deer         299
none              78
roe_deer          12
fox                3
red_deer,wolf      2
hare               2
wolf               1
Name: count, dtype: int64


In [61]:
print("NUMBER OF INDIVIDUALS")
print(b2["num_individuals"].value_counts(dropna=False))

NUMBER OF INDIVIDUALS
num_individuals
1     243
0      78
2      46
3+     30
Name: count, dtype: int64


In [62]:
print("WEATHER")
print(b2["weather"].value_counts(dropna=False))

WEATHER
weather
clear       181
sunny       108
overcast     72
rainy        36
Name: count, dtype: int64


In [63]:
activities = (
    b2["activities"]
    .str.split(",")
    .explode()
    .str.strip()
)

print("ACTIVITIES")
print(activities.value_counts(dropna=False))

ACTIVITIES
activities
foraging           285
vigilance          198
unknown            158
none                78
courtship           46
grooming            42
marking             36
camera_reaction     28
escaping            13
resting             13
chasing              7
playing              3
Name: count, dtype: int64


In [64]:
for col in ["activities", "species", "num_individuals"]:
    print(f"\n--- {col.upper()} UNIQUE VALUES ---")
    print(sorted(b2[col].dropna().unique()))


--- ACTIVITIES UNIQUE VALUES ---
['camera_reaction', 'camera_reaction,courtship,foraging', 'camera_reaction,courtship,foraging,grooming,marking,playing,resting,unknown,vigilance', 'camera_reaction,courtship,foraging,grooming,marking,unknown,vigilance', 'camera_reaction,foraging,grooming,unknown,vigilance', 'camera_reaction,foraging,unknown', 'camera_reaction,foraging,unknown,vigilance', 'camera_reaction,foraging,vigilance', 'camera_reaction,vigilance', 'chasing', 'chasing,courtship,escaping,foraging,grooming,marking,resting,unknown,vigilance', 'chasing,courtship,escaping,unknown', 'chasing,escaping', 'chasing,foraging,unknown,vigilance', 'chasing,foraging,vigilance', 'chasing,unknown,vigilance', 'courtship,escaping,foraging,grooming,playing,unknown,vigilance', 'courtship,escaping,foraging,grooming,vigilance', 'courtship,escaping,unknown', 'courtship,foraging', 'courtship,foraging,grooming,marking,resting,unknown,vigilance', 'courtship,foraging,grooming,marking,unknown,vigilance', 'cou

In [66]:
possible_background_terms = [
    "none",
    "background",
    "empty",
    "false",
    "no_animal",
    #"unknown",
    "0",
]

mask = pd.Series(False, index=b2.index)

for col in ["activities", "species", "num_individuals"]:
    col_values = b2[col].fillna("").str.lower()

    for term in possible_background_terms:
        mask |= col_values.str.contains(
            term,
            regex=False
        )

possible_background = b2[mask]

print("Potential background-like rows:", len(possible_background))

possible_background[
    [
        "event_id",
        "activities",
        "species",
        "num_individuals",
        "split"
    ]
].head(50)

Potential background-like rows: 78


,event_id,activities,species,num_individuals,split
18,E140,none,none,0,train
28,E166,none,none,0,train
29,E168,none,none,0,train
57,E204,none,none,0,train
58,E205,none,none,0,train
59,E208,none,none,0,train
60,E21,none,none,0,train
61,E210,none,none,0,train
64,E218,none,none,0,train
66,E221,none,none,0,train


In [67]:
b2_background = b2[
    (b2["activities"] == "none") &
    (b2["species"] == "none") &
    (b2["num_individuals"] == "0")
].copy()

print("B2 background events:", len(b2_background))
print(b2_background["event_id"].nunique())

B2 background events: 78
78


In [68]:
dense_event_summary = (
    dense_summary
    .groupby("event_id", as_index=False)
    .agg(
        num_videos=("file_id", "nunique"),
        videos_with_detection=("has_detection", "sum"),
        total_detections=("total_detections", "sum"),
        automatically_discarded=(
            "automatically_discarded",
            "sum"
        ),
        total_duration_s=("duration_s", "sum")
    )
)

dense_event_summary.head()

,event_id,num_videos,videos_with_detection,total_detections,automatically_discarded,total_duration_s
0,E10,1,1,2087,0,120.5
1,E103,10,9,2453,1,104.0
2,E104,25,23,4059,2,141.0
3,E105,56,54,21616,2,730.0
4,E106,21,13,3265,8,135.0


In [69]:
background_check = b2_background.merge(
    dense_event_summary,
    on="event_id",
    how="left"
)

print("B2 background events:", len(background_check))

print(
    "Matched to dense annotations:",
    background_check["num_videos"].notna().sum()
)

print(
    "Missing from dense annotations:",
    background_check["num_videos"].isna().sum()
)

B2 background events: 78
Matched to dense annotations: 78
Missing from dense annotations: 0


In [71]:
print("Events with ANY dense detection:")
print(
    (background_check["videos_with_detection"] > 0)
    .value_counts(dropna=False)
)

print("\nTotal detections:")
print(background_check["total_detections"].describe())

print("\nVideos per event:")
print(background_check["num_videos"].describe())

Events with ANY dense detection:
videos_with_detection
False    78
Name: count, dtype: int64

Total detections:
count    78.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: total_detections, dtype: float64

Videos per event:
count    78.000000
mean      2.179487
std       1.821542
min       1.000000
25%       1.000000
50%       1.500000
75%       2.000000
max      10.000000
Name: num_videos, dtype: float64


In [72]:
background_check["all_videos_discarded"] = (
    background_check["automatically_discarded"]
    == background_check["num_videos"]
)

print(
    background_check["all_videos_discarded"]
    .value_counts(dropna=False)
)

all_videos_discarded
True    78
Name: count, dtype: int64


In [73]:
b2_animal = b2[
    ~(
        (b2["activities"] == "none") &
        (b2["species"] == "none") &
        (b2["num_individuals"] == "0")
    )
].copy()

animal_check = b2_animal.merge(
    dense_event_summary,
    on="event_id",
    how="left"
)

print("Animal-labelled B2 events:", len(animal_check))

print(
    "Events with dense detections:",
    (animal_check["videos_with_detection"] > 0).sum()
)

print(
    "Events without dense detections:",
    (animal_check["videos_with_detection"] == 0).sum()
)

Animal-labelled B2 events: 319
Events with dense detections: 319
Events without dense detections: 0


In [74]:
b2_manifest = b2.copy()

background_mask = (
    (b2_manifest["activities"] == "none") &
    (b2_manifest["species"] == "none") &
    (b2_manifest["num_individuals"] == "0")
)

b2_manifest["binary_label"] = (~background_mask).astype(int)
b2_manifest["label_name"] = b2_manifest["binary_label"].map({
    0: "background",
    1: "animal"
})

print(b2_manifest["label_name"].value_counts())

label_name
animal        319
background     78
Name: count, dtype: int64


In [75]:
raw_event_summary = (
    raw_meta
    .groupby("event_id", as_index=False)
    .agg(
        num_videos=("file_id", "nunique"),
        total_duration_s=("duration", "sum"),

        site_ids=("site_id", lambda x: sorted(x.dropna().unique().tolist())),
        camera_ids=("cam_id", lambda x: sorted(x.dropna().unique().tolist())),

        file_ids=("file_id", lambda x: x.tolist()),
        file_paths=("file_path", lambda x: x.tolist()),
    )
)

raw_event_summary.head()

,event_id,num_videos,total_duration_s,site_ids,camera_ids,file_ids,file_paths
0,E10,1,120.5,[S1],[C1],[S1_C1_E10_V0031],[S1/C1/S1_C1_E10_V0031.mp4]
1,E103,10,104.0,[S1],"[C1, C2]","[S1_C1_E103_V0253, S1_C1_E103_V0254, S1_C1_E10...","[S1/C1/S1_C1_E103_V0253.mp4, S1/C1/S1_C1_E103_..."
2,E104,25,141.0,[S1],"[C1, C2]","[S1_C1_E104_V0262, S1_C1_E104_V0263, S1_C1_E10...","[S1/C1/S1_C1_E104_V0262.mp4, S1/C1/S1_C1_E104_..."
3,E105,56,730.0,[S1],"[C1, C2]","[S1_C1_E105_V0285, S1_C1_E105_V0286, S1_C1_E10...","[S1/C1/S1_C1_E105_V0285.mp4, S1/C1/S1_C1_E105_..."
4,E106,21,135.0,[S1],"[C1, C2]","[S1_C1_E106_V0311, S1_C1_E106_V0312, S1_C1_E10...","[S1/C1/S1_C1_E106_V0311.mp4, S1/C1/S1_C1_E106_..."


In [76]:
dense_event_summary = (
    dense_summary
    .groupby("event_id", as_index=False)
    .agg(
        videos_with_detection=("has_detection", "sum"),
        total_detections=("total_detections", "sum"),
        discarded_videos=("automatically_discarded", "sum")
    )
)

In [77]:
event_manifest = (
    b2_manifest
    .merge(
        raw_event_summary,
        on="event_id",
        how="left"
    )
    .merge(
        dense_event_summary,
        on="event_id",
        how="left"
    )
)

event_manifest["all_videos_discarded"] = (
    event_manifest["discarded_videos"]
    == event_manifest["num_videos"]
)

print(event_manifest.shape)

event_manifest.head()

(397, 19)


,event_id,feature_path,activities,species,weather,num_individuals,split,binary_label,label_name,num_videos,total_duration_s,site_ids,camera_ids,file_ids,file_paths,videos_with_detection,total_detections,discarded_videos,all_videos_discarded
0,E10,E10/E10_dtci_r11.npy,"foraging,unknown,vigilance",roe_deer,overcast,3+,train,1,animal,1,120.5,[S1],[C1],[S1_C1_E10_V0031],[S1/C1/S1_C1_E10_V0031.mp4],1,2087,0,False
1,E103,E103/E103_dtci_r11.npy,"foraging,unknown,vigilance",red_deer,clear,1,train,1,animal,10,104.0,[S1],"[C1, C2]","[S1_C1_E103_V0253, S1_C1_E103_V0254, S1_C1_E10...","[S1/C1/S1_C1_E103_V0253.mp4, S1/C1/S1_C1_E103_...",9,2453,1,False
2,E104,E104/E104_dtci_r11.npy,"courtship,foraging,grooming,vigilance",red_deer,clear,1,train,1,animal,25,141.0,[S1],"[C1, C2]","[S1_C1_E104_V0262, S1_C1_E104_V0263, S1_C1_E10...","[S1/C1/S1_C1_E104_V0262.mp4, S1/C1/S1_C1_E104_...",23,4059,2,False
3,E105,E105/E105_dtci_r11.npy,"courtship,foraging,marking,unknown,vigilance",red_deer,clear,1,train,1,animal,56,730.0,[S1],"[C1, C2]","[S1_C1_E105_V0285, S1_C1_E105_V0286, S1_C1_E10...","[S1/C1/S1_C1_E105_V0285.mp4, S1/C1/S1_C1_E105_...",54,21616,2,False
4,E106,E106/E106_dtci_r11.npy,"foraging,unknown,vigilance",red_deer,clear,1,train,1,animal,21,135.0,[S1],"[C1, C2]","[S1_C1_E106_V0311, S1_C1_E106_V0312, S1_C1_E10...","[S1/C1/S1_C1_E106_V0311.mp4, S1/C1/S1_C1_E106_...",13,3265,8,False


In [78]:
print("B2 events:", len(event_manifest))

print(
    "Missing raw videos:",
    event_manifest["num_videos"].isna().sum()
)

print(
    "Missing dense annotations:",
    event_manifest["total_detections"].isna().sum()
)

print(
    "Unique events:",
    event_manifest["event_id"].nunique()
)

B2 events: 397
Missing raw videos: 0
Missing dense annotations: 0
Unique events: 397


In [87]:
pd.crosstab(
    event_manifest["label_name"],
    event_manifest["videos_with_detection"]
)

videos_with_detection,0,1,2,3,4,5,6,7,8,9,...,23,24,25,27,29,38,43,45,54,84
label_name,,,,,,,,,,,,,,,,,,,,,
animal,0,109,55,46,21,12,15,3,11,4,...,1,3,1,2,2,1,1,1,1,1
background,78,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [86]:
event_summary = (
    event_manifest
    .groupby("label_name")
    .agg(
        events=("event_id", "count"),
        total_videos=("num_videos", "sum"),
        mean_videos_per_event=("num_videos", "mean"),
        median_videos_per_event=("num_videos", "median"),
        min_videos=("num_videos", "min"),
        max_videos=("num_videos", "max"),
        total_duration_s=("total_duration_s", "sum"),
        mean_duration_s=("total_duration_s", "mean"),
        median_duration_s=("total_duration_s", "median"),
        min_duration_s=("total_duration_s", "min"),
        max_duration_s=("total_duration_s", "max"),
    )
    .round(2)
)

event_summary

,events,total_videos,mean_videos_per_event,median_videos_per_event,min_videos,max_videos,total_duration_s,mean_duration_s,median_duration_s,min_duration_s,max_duration_s
label_name,,,,,,,,,,,
animal,319,2214,6.94,3.0,1,123,38584.93,120.96,61.50,2.0,786.5
background,78,170,2.18,1.5,1,10,12445.00,159.55,120.75,60.5,605.0


In [83]:
site_distribution = (
    event_manifest[
        ["event_id", "label_name", "site_ids"]
    ]
    .explode("site_ids")
    .groupby(["site_ids", "label_name"])
    .size()
    .unstack(fill_value=0)
)

site_distribution


label_name,animal,background
site_ids,,
S1,115,18
S2,74,3
S3,130,57


In [84]:
camera_distribution = (
    event_manifest[
        ["event_id", "label_name", "site_ids", "camera_ids"]
    ]
    .explode("site_ids")
    .explode("camera_ids")
    .groupby(["site_ids", "camera_ids", "label_name"])
    .size()
    .unstack(fill_value=0)
)

camera_distribution

label_name           animal  background
site_ids camera_ids                    
S1       C1              90          17
         C2              89           1
         C3              20           2
S2       C1              54           2
         C2              53           1
         C3              54           0
S3       C1               6           1
         C2             127          20
         C3              25          43

In [85]:
MANIFEST_PATH = (
    Path("metadata") /
    "benchmark_2_event_manifest.csv"
)

event_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

print("Saved:", MANIFEST_PATH)

Saved: metadata\benchmark_2_event_manifest.csv


In [89]:
positive_event_ids = set(
    event_manifest.loc[
        event_manifest["label_name"] == "animal",
        "event_id"
    ]
)

print("Positive B2 events:", len(positive_event_ids))

Positive B2 events: 319


In [90]:
VISIBLE_GAP_TOLERANCE_S = 0.5

visible_intervals = []

with zipfile.ZipFile(DENSE_ZIP) as z:

    for json_path in dense_json_files:

        with z.open(json_path) as f:
            annotation = json.load(f)

        info = annotation["info"]

        # Only B2 animal events
        if info["event_id"] not in positive_event_ids:
            continue

        fps = info["fps"]

        detected_frames = [
            frame["frame_id"]
            for frame in annotation["frames"]
            if len(frame["detections"]) > 0
        ]

        if not detected_frames:
            continue

        max_gap_frames = int(
            VISIBLE_GAP_TOLERANCE_S * fps
        )

        start_frame = detected_frames[0]
        previous_frame = detected_frames[0]

        for frame_id in detected_frames[1:]:

            if frame_id - previous_frame > max_gap_frames + 1:

                start_s = start_frame / fps
                end_s = (previous_frame + 1) / fps

                visible_intervals.append({
                    "event_id": info["event_id"],
                    "file_id": info["file_id"],
                    "site_id": info["site_id"],
                    "cam_id": info["cam_id"],
                    "start_s": start_s,
                    "end_s": end_s,
                    "visible_duration_s": end_s - start_s,
                    "video_duration_s": info["duration_s"],
                })

                start_frame = frame_id

            previous_frame = frame_id

        # Final interval
        start_s = start_frame / fps
        end_s = (previous_frame + 1) / fps

        visible_intervals.append({
            "event_id": info["event_id"],
            "file_id": info["file_id"],
            "site_id": info["site_id"],
            "cam_id": info["cam_id"],
            "start_s": start_s,
            "end_s": end_s,
            "visible_duration_s": end_s - start_s,
            "video_duration_s": info["duration_s"],
        })

visible_intervals = pd.DataFrame(visible_intervals)

print("Visible intervals:", len(visible_intervals))
print("Events represented:", visible_intervals["event_id"].nunique())
print("Videos represented:", visible_intervals["file_id"].nunique())

visible_intervals.head()

Visible intervals: 1681
Events represented: 319
Videos represented: 1639


,event_id,file_id,site_id,cam_id,start_s,end_s,visible_duration_s,video_duration_s
0,E103,S1_C1_E103_V0253,S1,C1,0.033333,20.500000,20.466667,20.5
1,E103,S1_C1_E103_V0254,S1,C1,0.000000,20.500000,20.500000,20.5
2,E103,S1_C1_E103_V0255,S1,C1,0.000000,11.166667,11.166667,11.5
3,E103,S1_C1_E103_V0256,S1,C1,0.000000,3.966667,3.966667,4.0
4,E103,S1_C1_E103_V0257,S1,C1,0.000000,7.600000,7.600000,8.0


In [91]:
interval_stats = pd.Series({
    "intervals": len(visible_intervals),
    "mean_s": visible_intervals["visible_duration_s"].mean(),
    "median_s": visible_intervals["visible_duration_s"].median(),
    "min_s": visible_intervals["visible_duration_s"].min(),
    "p25_s": visible_intervals["visible_duration_s"].quantile(0.25),
    "p75_s": visible_intervals["visible_duration_s"].quantile(0.75),
    "p90_s": visible_intervals["visible_duration_s"].quantile(0.90),
    "max_s": visible_intervals["visible_duration_s"].max(),
}).round(2)

interval_stats

intervals    1681.00
mean_s         14.91
median_s        9.17
min_s           0.10
p25_s           2.40
p75_s          20.50
p90_s          20.50
max_s         120.50
dtype: float64

In [92]:
window_lengths = [1, 2, 3, 5]

window_feasibility = []

for window_s in window_lengths:

    long_enough = (
        visible_intervals["visible_duration_s"] >= window_s
    )

    window_feasibility.append({
        "window_s": window_s,
        "intervals_long_enough": long_enough.sum(),
        "pct_intervals_long_enough":
            100 * long_enough.mean(),
        "events_with_interval_long_enough":
            visible_intervals.loc[
                long_enough,
                "event_id"
            ].nunique(),
        "pct_positive_events_covered":
            100 * (
                visible_intervals.loc[
                    long_enough,
                    "event_id"
                ].nunique()
                / len(positive_event_ids)
            )
    })

window_feasibility = (
    pd.DataFrame(window_feasibility)
    .round(1)
)

window_feasibility

,window_s,intervals_long_enough,pct_intervals_long_enough,events_with_interval_long_enough,pct_positive_events_covered
0,1,1672,99.5,319,100.0
1,2,1363,81.1,294,92.2
2,3,1174,69.8,282,88.4
3,5,1015,60.4,263,82.4


In [93]:
def can_fit_centered_window(row, window_s):
    midpoint = (row["start_s"] + row["end_s"]) / 2

    window_start = midpoint - window_s / 2
    window_end = midpoint + window_s / 2

    return (
        window_start >= 0
        and window_end <= row["video_duration_s"]
    )

In [94]:
centered_window_results = []

for window_s in window_lengths:

    can_fit = visible_intervals.apply(
        lambda row: can_fit_centered_window(
            row,
            window_s
        ),
        axis=1
    )

    covered_events = visible_intervals.loc[
        can_fit,
        "event_id"
    ].nunique()

    centered_window_results.append({
        "window_s": window_s,
        "valid_intervals": can_fit.sum(),
        "events_covered": covered_events,
        "pct_positive_events_covered":
            100 * covered_events / len(positive_event_ids)
    })

centered_window_results = (
    pd.DataFrame(centered_window_results)
    .round(1)
)

centered_window_results

,window_s,valid_intervals,events_covered,pct_positive_events_covered
0,1,1677,319,100.0
1,2,1375,295,92.5
2,3,1191,283,88.7
3,5,1033,266,83.4


In [95]:
WINDOW_LENGTHS = [1, 2, 3]


def fit_window_around_interval(
    interval_start,
    interval_end,
    video_duration,
    window_s
):

    if video_duration < window_s:
        return None

    midpoint = (interval_start + interval_end) / 2

    start = midpoint - window_s / 2

    # Shift to stay within video boundaries
    start = max(0, start)
    start = min(start, video_duration - window_s)

    end = start + window_s

    return start, end

In [96]:
positive_candidates = []

for _, row in visible_intervals.iterrows():

    for window_s in WINDOW_LENGTHS:

        window = fit_window_around_interval(
            row["start_s"],
            row["end_s"],
            row["video_duration_s"],
            window_s
        )

        if window is None:
            continue

        start_s, end_s = window

        # How much of this audio window overlaps
        # with actual visual animal presence?
        visible_overlap_s = max(
            0,
            min(end_s, row["end_s"])
            - max(start_s, row["start_s"])
        )

        positive_candidates.append({
            "event_id": row["event_id"],
            "file_id": row["file_id"],
            "window_s": window_s,
            "start_s": start_s,
            "end_s": end_s,
            "visible_overlap_s": visible_overlap_s,
            "visible_fraction": visible_overlap_s / window_s,
            "window_source": "animal_detection"
        })

positive_candidates = pd.DataFrame(
    positive_candidates
)

In [97]:
positive_candidates = (
    positive_candidates
    .drop_duplicates(
        subset=[
            "event_id",
            "file_id",
            "window_s",
            "start_s",
            "end_s"
        ]
    )
    .reset_index(drop=True)
)

print("Positive candidate windows:", len(positive_candidates))

Positive candidate windows: 4518


In [98]:
positive_coverage = []

for window_s in WINDOW_LENGTHS:

    subset = positive_candidates[
        positive_candidates["window_s"] == window_s
    ]

    covered = subset["event_id"].nunique()

    positive_coverage.append({
        "window_s": window_s,
        "candidate_windows": len(subset),
        "events_covered": covered,
        "pct_animal_events":
            100 * covered / len(positive_event_ids)
    })

pd.DataFrame(positive_coverage).round(1)

,window_s,candidate_windows,events_covered,pct_animal_events
0,1,1681,319,100.0
1,2,1534,311,97.5
2,3,1303,298,93.4


In [99]:
background_event_ids = set(
    event_manifest.loc[
        event_manifest["label_name"] == "background",
        "event_id"
    ]
)

background_videos = raw_meta[
    raw_meta["event_id"].isin(background_event_ids)
].copy()

In [100]:
background_candidates = []

for _, row in background_videos.iterrows():

    duration = row["duration"]

    for window_s in WINDOW_LENGTHS:

        if duration < window_s:
            continue

        max_start = duration - window_s

        starts = np.linspace(
            0,
            max_start,
            num=3
        )

        # Avoid duplicates for videos almost exactly window_s long
        starts = np.unique(
            np.round(starts, 6)
        )

        for start_s in starts:

            background_candidates.append({
                "event_id": row["event_id"],
                "file_id": row["file_id"],
                "window_s": window_s,
                "start_s": float(start_s),
                "end_s": float(start_s + window_s),
                "visible_overlap_s": 0.0,
                "visible_fraction": 0.0,
                "window_source": "background"
            })

background_candidates = pd.DataFrame(
    background_candidates
)

print(
    "Background candidate windows:",
    len(background_candidates)
)

Background candidate windows: 1530


In [101]:
window_candidates = pd.concat(
    [
        positive_candidates,
        background_candidates
    ],
    ignore_index=True
)

In [102]:
file_metadata = (
    raw_meta[
        [
            "file_id",
            "file_path",
            "site_id",
            "cam_id",
            "duration"
        ]
    ]
    .drop_duplicates("file_id")
)

window_candidates = window_candidates.merge(
    file_metadata,
    on="file_id",
    how="left"
)

In [103]:
event_metadata = event_manifest[
    [
        "event_id",
        "binary_label",
        "label_name",
        "species",
        "activities",
        "num_individuals",
        "split"
    ]
].drop_duplicates("event_id")

window_candidates = window_candidates.merge(
    event_metadata,
    on="event_id",
    how="left"
)

In [104]:
candidate_summary = (
    window_candidates
    .groupby(["window_s", "label_name"])
    .agg(
        windows=("file_id", "size"),
        events=("event_id", "nunique"),
        files=("file_id", "nunique")
    )
)

candidate_summary

windows  events  files
window_s label_name                        
1        animal         1681     319   1639
         background      510      78    170
2        animal         1534     311   1492
         background      510      78    170
3        animal         1303     298   1261
         background      510      78    170

In [105]:
MAX_WINDOWS_PER_EVENT = 3


def capped_sample(df, n=3, seed=42):

    sampled = []

    for i, (_, group) in enumerate(
        df.groupby(
            ["event_id", "window_s"],
            sort=True
        )
    ):
        sampled.append(
            group.sample(
                n=min(n, len(group)),
                random_state=seed + i
            )
        )

    return pd.concat(
        sampled,
        ignore_index=True
    )


window_manifest = capped_sample(
    window_candidates,
    n=MAX_WINDOWS_PER_EVENT
)

In [106]:
window_manifest = (
    window_manifest
    .sort_values(
        [
            "split",
            "event_id",
            "window_s",
            "start_s"
        ]
    )
    .reset_index(drop=True)
)

print("Final windows:", len(window_manifest))

Final windows: 2716


In [107]:
final_summary = (
    window_manifest
    .groupby(
        ["split", "window_s", "label_name"]
    )
    .agg(
        windows=("file_id", "size"),
        events=("event_id", "nunique"),
        files=("file_id", "nunique")
    )
)

final_summary

windows  events  files
split window_s label_name                        
test  1        animal          127      58    121
               background       84      28     45
      2        animal          123      57    118
               background       84      28     49
      3        animal          121      55    115
               background       84      28     47
train 1        animal          569     261    556
               background      150      50     79
      2        animal          551     254    542
               background      150      50     77
      3        animal          523     243    514
               background      150      50     77

In [108]:
assert (
    window_manifest
    .groupby("event_id")["split"]
    .nunique()
    .max()
    == 1
)

print("No event leakage between official splits.")

No event leakage between official splits.


In [109]:
CANDIDATE_PATH = (
    Path("metadata") /
    "benchmark_2_audio_window_candidates.csv"
)

WINDOW_MANIFEST_PATH = (
    Path("metadata") /
    "benchmark_2_audio_window_manifest.csv"
)

window_candidates.to_csv(
    CANDIDATE_PATH,
    index=False
)

window_manifest.to_csv(
    WINDOW_MANIFEST_PATH,
    index=False
)

print("Saved:", CANDIDATE_PATH)
print("Saved:", WINDOW_MANIFEST_PATH)

Saved: metadata\benchmark_2_audio_window_candidates.csv
Saved: metadata\benchmark_2_audio_window_manifest.csv


In [113]:
WINDOW_LENGTHS = [1.0, 2.0, 3.0]
MAX_ANCHORS_PER_EVENT = 3
RANDOM_SEED = 42

In [114]:
positive_anchors = visible_intervals[
    [
        "event_id",
        "file_id",
        "start_s",
        "end_s"
    ]
].copy()

positive_anchors["anchor_s"] = (
    positive_anchors["start_s"]
    + positive_anchors["end_s"]
) / 2

positive_anchors["anchor_source"] = "animal_detection"

positive_anchors = positive_anchors[
    [
        "event_id",
        "file_id",
        "anchor_s",
        "anchor_source"
    ]
].drop_duplicates()

print("Positive anchor candidates:", len(positive_anchors))
print("Positive events:", positive_anchors["event_id"].nunique())

Positive anchor candidates: 1681
Positive events: 319


In [115]:
background_event_ids = set(
    event_manifest.loc[
        event_manifest["label_name"] == "background",
        "event_id"
    ]
)

background_videos = raw_meta[
    raw_meta["event_id"].isin(background_event_ids)
].copy()

background_records = []

for _, row in background_videos.iterrows():

    duration = float(row["duration"])

    if duration <= 0:
        continue

    for fraction in [0.25, 0.50, 0.75]:

        background_records.append({
            "event_id": row["event_id"],
            "file_id": row["file_id"],
            "anchor_s": duration * fraction,
            "anchor_source": "background"
        })

background_anchors = pd.DataFrame(background_records)

print("Background anchor candidates:", len(background_anchors))
print("Background events:", background_anchors["event_id"].nunique())

Background anchor candidates: 510
Background events: 78


In [116]:
anchor_candidates = pd.concat(
    [
        positive_anchors,
        background_anchors
    ],
    ignore_index=True
)

sampled_groups = []

for i, (_, group) in enumerate(
    anchor_candidates.groupby("event_id", sort=True)
):
    sampled_groups.append(
        group.sample(
            n=min(MAX_ANCHORS_PER_EVENT, len(group)),
            random_state=RANDOM_SEED + i
        )
    )

anchors = pd.concat(
    sampled_groups,
    ignore_index=True
)

print("Selected anchors:", len(anchors))
print("Events represented:", anchors["event_id"].nunique())

Selected anchors: 930
Events represented: 397


In [117]:
file_metadata = (
    raw_meta[
        [
            "file_id",
            "file_path",
            "site_id",
            "cam_id",
            "duration"
        ]
    ]
    .drop_duplicates("file_id")
)

anchors = anchors.merge(
    file_metadata,
    on="file_id",
    how="left"
)

In [118]:
event_metadata = event_manifest[
    [
        "event_id",
        "binary_label",
        "label_name",
        "species",
        "activities",
        "num_individuals",
        "split"
    ]
].drop_duplicates("event_id")

anchors = anchors.merge(
    event_metadata,
    on="event_id",
    how="left"
)

In [119]:
print("Missing video metadata:", anchors["duration"].isna().sum())
print("Missing labels:", anchors["label_name"].isna().sum())

Missing video metadata: 0
Missing labels: 0


In [120]:
def make_nested_windows(anchor_s, duration_s, lengths=(1.0, 2.0, 3.0)):

    max_window = max(lengths)

    # Outer 3-second logical window
    if duration_s >= max_window:
        outer_start = np.clip(
            anchor_s - max_window / 2,
            0,
            duration_s - max_window
        )
    else:
        # Entire short video lies inside this virtual window
        outer_start = np.clip(
            anchor_s - max_window / 2,
            duration_s - max_window,
            0
        )

    outer_end = outer_start + max_window

    result = {}

    for window_s in lengths:

        # Window can fit normally inside the video
        if duration_s >= window_s:

            min_start = max(
                0,
                outer_start
            )

            max_start = min(
                duration_s - window_s,
                outer_end - window_s
            )

            start_s = np.clip(
                anchor_s - window_s / 2,
                min_start,
                max_start
            )

            end_s = start_s + window_s

            pad_left = 0.0
            pad_right = 0.0

        # Video itself is shorter than requested window
        else:

            logical_start = np.clip(
                anchor_s - window_s / 2,
                duration_s - window_s,
                0
            )

            logical_end = logical_start + window_s

            start_s = 0.0
            end_s = duration_s

            pad_left = max(
                0.0,
                -logical_start
            )

            pad_right = max(
                0.0,
                logical_end - duration_s
            )

        result[int(window_s)] = {
            "start": float(start_s),
            "end": float(end_s),
            "pad_left": float(pad_left),
            "pad_right": float(pad_right)
        }

    return result

In [121]:
records = []

for _, row in anchors.iterrows():

    windows = make_nested_windows(
        anchor_s=float(row["anchor_s"]),
        duration_s=float(row["duration"])
    )

    record = row.to_dict()

    for window_s, values in windows.items():

        suffix = f"{window_s}s"

        record[f"start_{suffix}"] = values["start"]
        record[f"end_{suffix}"] = values["end"]

        record[f"pad_left_{suffix}"] = values["pad_left"]
        record[f"pad_right_{suffix}"] = values["pad_right"]

    records.append(record)

anchor_manifest = pd.DataFrame(records)

In [122]:
anchor_manifest = (
    anchor_manifest
    .sort_values(
        [
            "split",
            "event_id",
            "file_id",
            "anchor_s"
        ]
    )
    .reset_index(drop=True)
)

anchor_manifest["anchor_index"] = (
    anchor_manifest
    .groupby("event_id")
    .cumcount()
)

anchor_manifest["sample_id"] = (
    anchor_manifest["event_id"].astype(str)
    + "_"
    + anchor_manifest["anchor_index"].astype(str)
)

In [123]:
nested_ok = (
    (anchor_manifest["start_3s"] <= anchor_manifest["start_2s"]) &
    (anchor_manifest["start_2s"] <= anchor_manifest["start_1s"]) &
    (anchor_manifest["end_1s"] <= anchor_manifest["end_2s"]) &
    (anchor_manifest["end_2s"] <= anchor_manifest["end_3s"])
)

print("Nested windows:", nested_ok.all())

Nested windows: True


In [124]:
print("Samples:", len(anchor_manifest))
print("Events:", anchor_manifest["event_id"].nunique())

anchor_manifest.groupby(
    ["split", "label_name"]
).agg(
    samples=("sample_id", "count"),
    events=("event_id", "nunique"),
    files=("file_id", "nunique")
)

Samples: 930
Events: 397


samples  events  files
split label_name                        
test  animal          127      58    122
      background       84      28     50
train animal          569     261    556
      background      150      50     79

In [125]:
padding_summary = []

for window_s in [1, 2, 3]:

    suffix = f"{window_s}s"

    padded = (
        anchor_manifest[f"pad_left_{suffix}"]
        + anchor_manifest[f"pad_right_{suffix}"]
    ) > 0

    temp = anchor_manifest[
        ["split", "label_name"]
    ].copy()

    temp["padded"] = padded

    summary = (
        temp
        .groupby(["split", "label_name"])
        .agg(
            samples=("padded", "size"),
            padded=("padded", "sum")
        )
        .reset_index()
    )

    summary["pct_padded"] = (
        100 * summary["padded"] / summary["samples"]
    )

    summary["window_s"] = window_s

    padding_summary.append(summary)

padding_summary = pd.concat(
    padding_summary,
    ignore_index=True
)

padding_summary.round(1)

,split,label_name,samples,padded,pct_padded,window_s
0,test,animal,127,0,0.0,1
1,test,background,84,0,0.0,1
2,train,animal,569,0,0.0,1
3,train,background,150,0,0.0,1
4,test,animal,127,7,5.5,2
5,test,background,84,0,0.0,2
6,train,animal,569,35,6.2,2
7,train,background,150,0,0.0,2
8,test,animal,127,17,13.4,3
9,test,background,84,0,0.0,3


In [126]:
ANCHOR_MANIFEST_PATH = (
    Path("metadata") /
    "benchmark_2_audio_anchor_manifest.csv"
)

anchor_manifest.to_csv(
    ANCHOR_MANIFEST_PATH,
    index=False
)

print("Saved:", ANCHOR_MANIFEST_PATH)

Saved: metadata\benchmark_2_audio_anchor_manifest.csv


In [127]:
anchor_candidates_no_pad = anchor_candidates.merge(
    file_metadata[
        ["file_id", "duration"]
    ],
    on="file_id",
    how="left"
)

In [128]:
primary_candidates = anchor_candidates_no_pad[
    anchor_candidates_no_pad["duration"] >= 2.0
].copy()

print(
    "Events available:",
    primary_candidates["event_id"].nunique()
)

Events available: 389


In [129]:
sampled_groups = []

for i, (_, group) in enumerate(
    primary_candidates.groupby("event_id", sort=True)
):
    sampled_groups.append(
        group.sample(
            n=min(MAX_ANCHORS_PER_EVENT, len(group)),
            random_state=RANDOM_SEED + i
        )
    )

primary_anchors = pd.concat(
    sampled_groups,
    ignore_index=True
)

In [130]:
primary_anchors = (
    primary_anchors
    .drop(columns=["duration"], errors="ignore")
    .merge(
        file_metadata,
        on="file_id",
        how="left"
    )
    .merge(
        event_metadata,
        on="event_id",
        how="left"
    )
)

In [131]:
def make_nested_1s_2s(anchor_s, duration_s):

    # First place the outer 2-second window
    start_2s = np.clip(
        anchor_s - 1.0,
        0,
        duration_s - 2.0
    )

    end_2s = start_2s + 2.0

    # Then place the 1-second window inside it
    start_1s = np.clip(
        anchor_s - 0.5,
        start_2s,
        end_2s - 1.0
    )

    end_1s = start_1s + 1.0

    return start_1s, end_1s, start_2s, end_2s

In [132]:
windows = primary_anchors.apply(
    lambda row: make_nested_1s_2s(
        float(row["anchor_s"]),
        float(row["duration"])
    ),
    axis=1,
    result_type="expand"
)

windows.columns = [
    "start_1s",
    "end_1s",
    "start_2s",
    "end_2s"
]

primary_manifest = pd.concat(
    [
        primary_anchors.reset_index(drop=True),
        windows.reset_index(drop=True)
    ],
    axis=1
)

In [133]:
nested_ok = (
    (primary_manifest["start_2s"] <= primary_manifest["start_1s"]) &
    (primary_manifest["end_1s"] <= primary_manifest["end_2s"])
)

print("All nested:", nested_ok.all())

summary = (
    primary_manifest
    .groupby(["split", "label_name"])
    .agg(
        samples=("file_id", "size"),
        events=("event_id", "nunique"),
        files=("file_id", "nunique")
    )
)

summary

All nested: True


samples  events  files
split label_name                        
test  animal          123      57    117
      background       84      28     48
train animal          551     254    536
      background      150      50     78

In [134]:
all_events = set(event_manifest["event_id"])
primary_events = set(primary_manifest["event_id"])

missing_events = event_manifest[
    event_manifest["event_id"].isin(
        all_events - primary_events
    )
][
    [
        "event_id",
        "split",
        "label_name",
        "species",
        "num_videos",
        "total_duration_s"
    ]
]

print("Events excluded:", len(missing_events))
missing_events

Events excluded: 8


,event_id,split,label_name,species,num_videos,total_duration_s
34,E172,train,animal,red_deer,5,5.500000
72,E231,train,animal,red_deer,5,5.500000
74,E238,train,animal,red_deer,5,6.000000
81,E251,train,animal,red_deer,13,14.500000
263,E747,train,animal,red_deer,2,2.500000
272,E759,train,animal,red_deer,4,4.500000
293,E788,train,animal,red_deer,6,5.466667
319,E138,test,animal,red_deer,7,8.500000
